# adversary

> Bad agents for Phase 9: a liar (honest learner, false reports), a free-rider (takes reported value, never explores or shares) and a saboteur (crowds the top reported patch), for peer channels and for the lead; and the report log used for white-box detection.

Spec: `specs/multiagent/phase9/` (design-low-level section 2).

In [ ]:
#| default_exp adversary

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from collections import deque

import numpy as np

from explRL.harness import Blackboard, MessageBus
from explRL.harness_agents import CommBandit, LeadFollower
from explRL.team_agents import banditAgent

NAN = float('nan')


class ReportLog:
    "Mixin: logs each report as (location, value, own belief, t); own belief = mean of the agent's own stays there."
    def _belief(self, loc):
        r = self.rates[loc] if self.rates is not None else ()
        return float(np.mean(r)) if len(r) else 0.0

    def _log(self, loc, value):
        if not hasattr(self, 'reportLog'):
            self.reportLog = []
        self.reportLog.append((int(loc), float(value), self._belief(loc), float(self._t)))

    def whiteScore(self):
        "Mean distance between what the agent reported and what it believed (nan if it never reported)."
        log = getattr(self, 'reportLog', [])
        return float(np.mean([abs(v - b) for _, v, b, _ in log])) if log else NAN


class Liar(ReportLog, CommBandit):
    "Forages and learns honestly; reports -dangerCost where it harvested and c (best solo rate) where it did not."
    def learn(self, obs, target, reward, info):
        banditAgent.learn(self, obs, target, reward, info)
        if info['kind'] != 'stay':
            return
        w, loc = self._world, obs['location']
        lie = -w.dangerCost if info['harvest'] > 0 else w.c
        if isinstance(self.channel, Blackboard):
            self.channel.write(self.id, loc, lie, self._t)
            self._log(loc, lie)
        elif isinstance(self.channel, MessageBus) and self.channel.due(self.id, self._t):
            own = [(np.mean(r), j) for j, r in enumerate(self.rates) if len(r)]
            if own:
                j = min(own)[1]                                       # its worst own place, sold as the best
                self.channel.send(w, self.id, j, w.c, self._t)
                self._log(j, w.c)


class FreeRider(CommBandit):
    "Goes to the reported-positive location it rates best (no exploration bonus); stays put otherwise; never shares."
    def _reported(self):
        "Others' reports: mean per location (nan where nothing is reported)."
        L = self._world.L
        if isinstance(self.channel, Blackboard):
            S, C = self.channel.view(self.id, self._t)
        elif self.reports is not None:
            S = np.array([sum(r) for r in self.reports], float)
            C = np.array([len(r) for r in self.reports], float)
        else:
            return np.full(L, np.nan)
        return np.where(C > 0, S / np.where(C > 0, C, 1), np.nan)

    def act(self, obs):
        if self.rates is None:
            self.rates = [deque(maxlen=self.K) for _ in obs['travel']]
        ok = np.flatnonzero(np.nan_to_num(self._reported(), nan=-np.inf) > 0)
        if not len(ok):
            return obs['location']
        mean, _ = self._stats(obs)
        return int(ok[np.argmax(mean[ok])])

    def learn(self, obs, target, reward, info):
        banditAgent.learn(self, obs, target, reward, info)


class Saboteur(FreeRider):
    "Always goes to the location with the highest reported value and stays; never shares."
    def act(self, obs):
        if self.rates is None:
            self.rates = [deque(maxlen=self.K) for _ in obs['travel']]
        rep = self._reported()
        return int(np.nanargmax(rep)) if np.isfinite(rep).any() else obs['location']


class LeadLiar(LeadFollower):
    "Follows the lead's assignments and reports the liar's lie to it."
    def event(self, world, t):
        self._world = world
        return super().event(world, t)

    def learn(self, obs, target, reward, info):
        if info['kind'] == 'stay':
            w = self._world
            self.lead.report(self.id, obs['location'], -w.dangerCost if info['harvest'] > 0 else w.c, 1, self._t)


class LeadFreeRider(LeadFollower):
    "Follows the lead's assignments and never reports."
    def learn(self, obs, target, reward, info):
        pass


class LeadSaboteur(LeadFreeRider):
    "Goes where the lead has sent the most other followers; never reports."
    def act(self, obs):
        a = [x for k, x in enumerate(self.lead.assigned) if k != self.id and x is not None]
        return int(np.bincount(a).argmax()) if a else obs['location']


BAD = dict(liar=(Liar, LeadLiar), freerider=(FreeRider, LeadFreeRider), saboteur=(Saboteur, LeadSaboteur))

## Tests

In [ ]:
import numpy as np
from collections import deque
from fastcore.test import test_eq, test_close
from explRL.adversary import ReportLog, Liar, FreeRider, Saboteur, LeadLiar, LeadFreeRider, LeadSaboteur, BAD
from explRL.harness import Blackboard, MessageBus, Lead
from explRL.harness_runs import benchWorld
from explRL.multi import spawnRngs

def world(n):
    w = benchWorld(nPatches=1, nDanger=1, intake=(1.0,), patchSites=[0], dangerSites=[1], noise=0.0, Tmove=None)
    w.reset(0, n); return w

# liar on the blackboard: harvests at the patch (site 0) and reports -dangerCost; at the danger site (1) reports c;
# its own beliefs stay honest, and its white-box score is the gap between the two
w = world(2); w.loc = [0, 1]
bb = Blackboard(w.L, n=2)
liars = [Liar(bb), Liar(bb)]
for i, m in enumerate(liars): m.id = i
liars[0].event(w, 0.0); liars[1].event(w, 0.0)
S, C = bb.view(-1, 1.0)
test_close((S[0] / C[0], S[1] / C[1]), (-w.dangerCost, w.c))
test_close((liars[0].rates[0][0], liars[1].rates[1][0]), (w.c, -w.dangerCost))       # honest own beliefs
test_close(liars[0].whiteScore(), w.c + w.dangerCost)

# liar with messages: sends its worst own place as worth c
w = world(2)
bus = MessageBus(2, k=1, rngs=spawnRngs(1, 2))
lm = Liar(bus); lm.id, lm._world, lm._t = 0, w, 0.0
lm.rates = [deque() for _ in range(w.L)]; lm.rates[1].append(-2.6)
lm.learn({'location': 0}, 0, 0.5, {'kind': 'stay', 'harvest': 0.5, 'dur': 1.0})
test_eq([(j, round(v, 3)) for _, j, v, _ in bus.take(1)], [(1, w.c)])

# free-rider: only reported-positive locations, best combined estimate, stays put otherwise, never writes;
# saboteur: always the top reported location
w = world(3); w.loc = [10, 10, 10]
bb = Blackboard(w.L, n=3)
fr, sb = FreeRider(bb), Saboteur(bb)
for i, m in enumerate((fr, sb)): m.id, m._world, m._t = i, w, 0.0
obs = w.observe(0)
test_eq(fr.act(obs), 10); test_eq(sb.act(obs), 10)                                   # nothing reported: stay
bb.write(2, 3, 0.5, 0.0); bb.write(2, 5, 2.0, 0.0)
test_eq(sb.act(obs), 5); test_eq(fr.act(obs), 5)
fr.rates[5].extend([-2.6, -2.6, -2.6])                                               # its own bad experience at 5
test_eq(fr.act(obs), 3); test_eq(sb.act(obs), 5)
fr.learn({'location': 3}, 3, 0.5, {'kind': 'stay', 'harvest': 0.5, 'dur': 1.0})
test_eq(bb.stats[0]['written'], 0)
nc = FreeRider(None); nc.id, nc._world, nc._t = 0, w, 0.0
test_eq(nc.act(obs), 10)                                                             # no channel: stays put

# lead variants: the liar reports its lie, the free-rider stays silent, the saboteur crowds the most-assigned place
lead = Lead(w.L, 3, lam=0.0)
ll, lf, ls = LeadLiar(lead), LeadFreeRider(lead), LeadSaboteur(lead)
for i, m in enumerate((ll, lf, ls)): m.id, m._t, m._world = i, 0.0, w
ll.learn({'location': 0}, 0, 0.85, {'kind': 'stay', 'harvest': 0.85, 'dur': 1.0, 'nHere': 1})
S, C = lead.table.view(-1, 0.0); test_close(S[0] / C[0], -w.dangerCost)
lf.learn({'location': 0}, 0, 0.85, {'kind': 'stay', 'harvest': 0.85, 'dur': 1.0, 'nHere': 1})
test_eq(lead.stats[1]['written'], 0)
lead.assigned = [7, 4, None]
test_eq(ls.act(w.observe(2)), 4)                                                     # one each: ties to the lower index
lead.assigned = [7, 7, None]; test_eq(ls.act(w.observe(2)), 7)
test_eq(set(BAD), {'liar', 'freerider', 'saboteur'})

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()